# 1. Phase 3 objective and gate philosophy

Learn recurrent 2v2/3v2 team play, then improve cross-physics robustness without losing nominal competence. Gate A blocks training when the benchmark is permissive; Gate B blocks robustness work when nominal cooperation is absent; Gate C blocks final seeds when competence-constrained fine-tuning fails. Smoke results never authorize scientific training.

In [1]:
print('Phase 3 is staged, gated, and report-first. No cell silently launches training.')

Phase 3 is staged, gated, and report-first. No cell silently launches training.


# 2. Runtime variables

In [6]:
from pathlib import Path
import json, os, shutil, sys
REPOSITORY_URL = os.environ.get('ROBOSOCCER_REPOSITORY_URL', 'https://github.com/djdhillxn/football')
REPO_DIR = Path('/content/robot-soccer-transfer')
DRIVE_PROJECT = Path('/content/drive/MyDrive/RobotSoccerTransfer')
RUNS_ROOT = REPO_DIR / 'runs'
DEV_SEED = 3
NUM_ENVS = 16
CALIBRATION_EPISODES = 100
FINAL_CONFIRMATION_ENABLED = False

# 3. L4 and CUDA inspection

In [3]:
!nvidia-smi
import torch
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

Thu Jul 23 06:46:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 4. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


# 5. Clone or fast-forward pull

In [22]:
if (REPO_DIR / '.git').is_dir():
    %cd /content/robot-soccer-transfer
    !git pull --ff-only
    assert _exit_code == 0, 'Git pull failed; resolve the printed working-tree conflict before running later cells.'
else:
    assert REPOSITORY_URL, 'Set ROBOSOCCER_REPOSITORY_URL before cloning.'
    !git clone "$REPOSITORY_URL" /content/robot-soccer-transfer
    assert _exit_code == 0, 'Git clone failed.'
%cd /content/robot-soccer-transfer

/content/robot-soccer-transfer
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 16 (delta 6), reused 16 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 1.41 MiB | 15.03 MiB/s, done.
From https://github.com/djdhillxn/football
   1baea78..7bbec69  main       -> origin/main
Updating 1baea78..7bbec69
error: The following untracked working tree files would be overwritten by merge:
	reports/figures/canonical_transfer_gap.png
	reports/figures/method_success_by_suite.png
Please move or remove them before you merge.
Aborting
/content/robot-soccer-transfer


# 6. Install package

In [8]:
!python -u -m pip install -e '.[dev]' --no-deps
!python -u -m pip install pymunk==7.3.0 pettingzoo==1.26.1 gymnasium==1.2.1 tqdm==4.67.1

Obtaining file:///content/robot-soccer-transfer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for robosoccer-transfer (pyproject.toml) ... done
  Created wheel for robosoccer-transfer: filename=robosoccer_transfer-0.1.0-0.editable-py3-none-any.whl size=10494 sha256=8119a2e2c088012d9cbcaa7466406140b83205a1ee4f4bf615dad2e8ac2e4b78
  Stored in directory: /tmp/pip-ephem-wheel-cache-ocahurbz/wheels/3d/43/56/246f082290254c2be77c69e3c31ccdfe78a4ed07ed32620876
Successfully built robosoccer-transfer
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 66.4 MB/s eta 0:00:00
   

# 7. Restore selected Phase 2 artifacts

In [9]:
from robosoccer.artifacts import merge_artifact_directory, sync_artifacts_from_drive
sync_artifacts_from_drive(DRIVE_PROJECT, REPO_DIR, include_training_artifacts=False)
phase2_selected = ['20260722_002437_mappo_nominal_mappo_seed0', '20260722_020456_mappo_uniform_dr_mappo_seed0', '20260722_033215_mappo_failure_dr_mappo_seed0']
for run_name in phase2_selected:
    source = DRIVE_PROJECT / 'runs' / run_name
    if source.is_dir():
        merge_artifact_directory(source, RUNS_ROOT / run_name, include_training_artifacts=True)
for source in sorted((DRIVE_PROJECT / 'runs').glob('*phase3*')):
    metadata_path = source / 'run_metadata.json'
    if metadata_path.is_file() and json.loads(metadata_path.read_text()).get('status') == 'complete':
        merge_artifact_directory(source, RUNS_ROOT / source.name, include_training_artifacts=True)
def phase3_stage_run(stage):
    matches = []
    for run_dir in RUNS_ROOT.iterdir():
        metadata_path = run_dir / 'run_metadata.json'
        if not metadata_path.is_file():
            continue
        metadata = json.loads(metadata_path.read_text())
        active = metadata.get('resolved_configuration', {}).get('phase3', {}).get('active_stage')
        if metadata.get('status') == 'complete' and metadata.get('experiment_name') == 'phase3_recurrent_nominal' and active == stage:
            matches.append(run_dir)
    if not matches:
        raise FileNotFoundError(f'No complete {stage} run is restored.')
    return max(matches, key=lambda path: path.name)
print('Selected Phase 2 and completed Phase 3 checkpoints restored; lightweight context restored for all runs.')

Selected Phase 2 and completed Phase 3 checkpoints restored; lightweight context restored for all runs.


# 8. Workspace artifact audit

In [10]:
!python -u -m scripts.audit_workspace --repository-root .

{
  "schema_version": 1,
  "repository": "/content/robot-soccer-transfer",
  "run_count": 17,
  "runs": {
    "20260721_055932_smoke_test_mappo_seed0": {
      "path": "runs/20260721_055932_smoke_test_mappo_seed0",
      "status": "complete",
      "experiment_name": "smoke_test",
      "seed": 0,
      "config_present": true
    },
    "20260721_055953_baselines_heuristic_seed0": {
      "path": "runs/20260721_055953_baselines_heuristic_seed0",
      "status": "complete",
      "experiment_name": "base",
      "seed": 0,
      "config_present": true
    },
    "20260721_060143_ippo_nominal_ippo_seed0": {
      "path": "runs/20260721_060143_ippo_nominal_ippo_seed0",
      "status": "complete",
      "experiment_name": "ippo_nominal",
      "seed": 0,
      "config_present": true
    },
    "20260721_095003_smoke_test_mappo_seed0": {
      "path": "runs/20260721_095003_smoke_test_mappo_seed0",
      "status": "complete",
      "experiment_name": "smoke_test",
      "seed": 0,
      "con

# 9. Compile updated reports

In [11]:
!python -u -m scripts.export_phase2_report
if shutil.which('latexmk'):
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex
else:
    print('latexmk unavailable; sources and generated tables remain authoritative.')

reports/generated_results.tex
latexmk unavailable; sources and generated tables remain authoritative.


# 10. Run Ruff and pytest

In [12]:
!ruff check robosoccer scripts tests
!python -u -m pytest -q

/bin/bash: line 1: ruff: command not found
........................................................................ [ 87%]
..........                                                               [100%]
=============================== warnings summary ===============================
tests/test_core.py::test_smoke_training_creates_required_artifacts
tests/test_core.py::test_metadata_is_valid_json_after_smoke
tests/test_core.py::test_run_pointer_created
  /usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

tests/test_phase3.py::test_recurrent_checkpoint_round_trip_and_export
  /usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:328: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This me

# 11. Run Phase 3 smoke test

In [13]:
from robosoccer.artifacts import resolve_run_pointer, sync_run_to_drive
!python -u -m scripts.train --config configs/phase3_smoke.yaml --seed 3
assert _exit_code == 0
smoke_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_smoke.txt', REPO_DIR)
sync_run_to_drive(smoke_run, DRIVE_PROJECT)
print(smoke_run)

Phase3 stage_a:  75% 96/128 [00:00<00:00, 165.49env-step/s, coop=0.00, ent=1.71, gpu=0M, kl=0.000, lr=3.7e-05, ret=-3.96, sps=241, step=112/128, success=0.00, val=-]WARNING: Learning-rate floor active: actor 3.00e-05 critic 3.00e-05
Phase3 stage_a: 100% 128/128 [00:00<00:00, 174.33env-step/s, coop=0.00, ent=1.71, gpu=0M, kl=0.000, lr=3.0e-05, ret=0.00, sps=251, step=128/128, success=0.00, val=0.00]
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

  warnings.warn(

  warnings.warn(

  if self.input_size != input.size(-1):

  if hx.size() != expected_hidden_size:

INFO: Completed run: /content/robot-soccer-transfer/runs/20260723_065748_phase3_smoke_mappo_seed3
/content/robot-soccer-transfer/runs/20260723_065748_phase3_smoke_mappo_seed3


# 12. Calibrate abstract and Pymunk benchmark hardness

In [14]:
CALIBRATION_DIR = RUNS_ROOT / f'phase3_calibration_seed{DEV_SEED}'
!python -u -m scripts.calibrate_phase3 --config configs/phase3_base.yaml --output-dir "$CALIBRATION_DIR" --episodes "$CALIBRATION_EPISODES" --seed-base 310000
assert _exit_code == 0
merge_artifact_directory(CALIBRATION_DIR, DRIVE_PROJECT / 'runs' / CALIBRATION_DIR.name, include_training_artifacts=True)

{
  "schema_version": 1,
  "smoke": false,
  "training_authorized": true,
  "passed": true,
  "checks": {
    "required_cells": {
      "passed": true,
      "observed": [],
      "criterion": "all required method/simulator/scenario cells exist"
    },
    "episode_count": {
      "passed": true,
      "observed": [],
      "criterion": "every required cell has at least 100 episodes"
    },
    "finite_metrics": {
      "passed": true,
      "observed": [],
      "criterion": "all required cell metrics are finite"
    },
    "random_pymunk_hard": {
      "passed": true,
      "observed": 0.01,
      "criterion": "random Pymunk open success <= 0.05"
    },
    "double_chase_pymunk_hard": {
      "passed": true,
      "observed": 0.03,
      "criterion": "double-chase Pymunk open success <= 0.20"
    },
    "role_pymunk_band": {
      "passed": true,
      "observed": 0.26,
      "criterion": "role Pymunk open success in [0.20, 0.55]"
    },
    "role_advantage": {
      "passed": true,


2

# 13. Display benchmark-gate result

In [15]:
CALIBRATION_SUMMARY = CALIBRATION_DIR / 'calibration_summary.json'
calibration = json.loads(CALIBRATION_SUMMARY.read_text())
display(calibration['checks'])
assert calibration['training_authorized'], 'Gate A failed: do not train or weaken thresholds.'

{'double_chase_pymunk_hard': {'criterion': 'double-chase Pymunk open success <= 0.20',
  'observed': 0.03,
  'passed': True},
 'episode_count': {'criterion': 'every required cell has at least 100 episodes',
  'observed': [],
  'passed': True},
 'finite_metrics': {'criterion': 'all required cell metrics are finite',
  'observed': [],
  'passed': True},
 'no_saturation': {'criterion': 'no required cell success > 0.85',
  'observed': {},
  'passed': True},
 'nontrivial_duration': {'criterion': 'every successful required cell median time >= 8.0s',
  'observed': {'double_chase__abstract__phase3_2v2_open': 33.2,
   'double_chase__pymunk__phase3_2v2_open': 26.0,
   'random__abstract__phase3_2v2_open': 10.4,
   'random__abstract__phase3_2v2_pass_required': 25.800000000000004,
   'random__pymunk__phase3_2v2_open': 13.600000000000001,
   'random__pymunk__phase3_2v2_pass_required': 14.8,
   'role_based__abstract__phase3_2v2_open': 17.4,
   'role_based__abstract__phase3_2v2_pass_required': 16.0,
 

# 14. Run throughput benchmark

In [16]:
THROUGHPUT_PATH = RUNS_ROOT / 'phase3_throughput.json'
!python -u -m scripts.benchmark_throughput --config configs/phase3_recurrent_nominal.yaml --num-envs 32 64 128 256 512 --updates 2 --output "$THROUGHPUT_PATH"
merge_artifact_directory(RUNS_ROOT, DRIVE_PROJECT / 'runs', skipped_relative_paths=[], include_training_artifacts=False)

{
  "schema_version": 1,
  "scientific_result": false,
  "rows": [
    {
      "num_envs": 32,
      "updates": 2,
      "rollout_steps": 64,
      "actor_minibatch_size": 2048,
      "recurrent_sequence_length": 32,
      "reset_seconds": 1.4369203310000103,
      "rollout_seconds": 12.203690773000062,
      "rollout_transfer_seconds": 0.04150645800064012,
      "actor_inference_seconds": 1.102656367001373,
      "environment_stepping_seconds": 10.934880587999714,
      "actor_optimization_seconds": 0.9295333280001614,
      "critic_optimization_seconds": 0.5806231480000861,
      "ppo_update_seconds": 1.5101564760002475,
      "effective_transitions_per_second": 298.67621577151397,
      "agent_steps_per_second": 896.028647314542,
      "transitions": 4096,
      "cpu_count": 12,
      "cuda_allocated_bytes": 34117632,
      "cuda_reserved_bytes": 88080384,
      "cuda_peak_memory_bytes": 74493952,
      "backend": "authoritative_lane_physics_batched_policy"
    },
    {
      "num_e

4

# 15. Select `num_envs` from the benchmark

In [17]:
throughput = json.loads(THROUGHPUT_PATH.read_text())
best = max(throughput['rows'], key=lambda row: row['effective_transitions_per_second'])
NUM_ENVS = int(best['num_envs'])
print({'selected_num_envs': NUM_ENVS, 'measurement': best})

{'selected_num_envs': 128, 'measurement': {'actor_inference_seconds': 0.3283427999995183, 'actor_minibatch_size': 2048, 'actor_optimization_seconds': 2.2421108349999486, 'agent_steps_per_second': 1012.3605597711372, 'backend': 'authoritative_lane_physics_batched_policy', 'cpu_count': 12, 'critic_optimization_seconds': 2.0707000789998347, 'cuda_allocated_bytes': 35481600, 'cuda_peak_memory_bytes': 75876352, 'cuda_reserved_bytes': 88080384, 'effective_transitions_per_second': 337.4535199237124, 'environment_stepping_seconds': 43.48821990900046, 'num_envs': 128, 'ppo_update_seconds': 4.312810913999783, 'recurrent_sequence_length': 32, 'reset_seconds': 0.2509321780000846, 'rollout_seconds': 44.23906077400011, 'rollout_steps': 64, 'rollout_transfer_seconds': 0.06470315100000334, 'transitions': 16384, 'updates': 2}}


# 16. Train recurrent nominal Stage A

In [18]:
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_a --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_a_run = phase3_stage_run('stage_a')
sync_run_to_drive(stage_a_run, DRIVE_PROJECT)

2026-07-23 07:08:27.035426: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-23 07:08:27.105960: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-23 07:08:28,835:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

Phase3 stage_a:  90% 450560/500000 [23:24<02:41, 306.82env-step/s, coop=0.68, ent=0.75, gpu=33M

{'destination': '/content/drive/MyDrive/RobotSoccerTransfer/runs/20260723_070824_phase3_recurrent_nominal_mappo_seed3',
 'copied_files': 20,
 'status': 'complete'}

# 17. Evaluate Stage A

In [19]:
stage_a_run = phase3_stage_run('stage_a')
!python -u -m scripts.evaluate_phase3 --run-dir "$stage_a_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_open --episodes 100 --seed-base 330000 --prefix stage_a_2v2_open
sync_run_to_drive(stage_a_run, DRIVE_PROJECT)

{
  "schema_version": 1,
  "checkpoint": "/content/robot-soccer-transfer/runs/20260723_070824_phase3_recurrent_nominal_mappo_seed3/models/best_nominal_checkpoint.pt",
  "simulator": "pymunk",
  "scenario": "phase3_2v2_open",
  "profile": "nominal",
  "episodes": 100,
  "success_rate": 0.43,
  "cooperative_success_rate": 0.43,
  "mean_return": 5.756914286133073,
  "pass_completion_rate": 0.9773210489014883,
  "mean_meaningful_action_count": 5.81,
  "median_success_time": 8.0
}


{'destination': '/content/drive/MyDrive/RobotSoccerTransfer/runs/20260723_070824_phase3_recurrent_nominal_mappo_seed3',
 'copied_files': 2,
 'status': 'complete'}

# 18. Train Stage B from Stage A

In [24]:
stage_a_run = phase3_stage_run('stage_a')
stage_a_checkpoint = stage_a_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_b --resume "$stage_a_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_b_run = phase3_stage_run('stage_b')
sync_run_to_drive(stage_b_run, DRIVE_PROJECT)

ERROR: Training failed: RNG state must be a torch.ByteTensor
Traceback (most recent call last):
  File "/content/robot-soccer-transfer/robosoccer/training.py", line 1016, in run_training
    trainer = RecurrentMAPPOTrainer(
              ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/robot-soccer-transfer/robosoccer/recurrent.py", line 436, in __init__
    self.load_checkpoint(resume_path, warm_start=False)
  File "/content/robot-soccer-transfer/robosoccer/recurrent.py", line 1268, in load_checkpoint
    torch.set_rng_state(checkpoint["torch_cpu_rng_state"])
  File "/usr/local/lib/python3.12/dist-packages/torch/random.py", line 19, in set_rng_state
    default_generator.set_state(new_state)
TypeError: RNG state must be a torch.ByteTensor
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/robot-soccer-transfer/scripts/train.py", line 55, in <module>
    main()
  File "/content/robot-soccer

AssertionError: 

# 19. Evaluate pass-required cooperation

In [ ]:
stage_b_run = phase3_stage_run('stage_b')
!python -u -m scripts.evaluate_phase3 --run-dir "$stage_b_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_pass_required --episodes 100 --seed-base 331000 --prefix stage_b_pass_required
sync_run_to_drive(stage_b_run, DRIVE_PROJECT)

# 20. Train Stage C from Stage B

In [ ]:
stage_b_run = phase3_stage_run('stage_b')
stage_b_checkpoint = stage_b_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_c --resume "$stage_b_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_c_run = phase3_stage_run('stage_c')
sync_run_to_drive(stage_c_run, DRIVE_PROJECT)

# 21. Train Stage D from Stage C

In [ ]:
stage_c_run = phase3_stage_run('stage_c')
stage_c_checkpoint = stage_c_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_d --resume "$stage_c_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
nominal_run = phase3_stage_run('stage_d')
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 22. Evaluate full nominal curriculum

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.evaluate_phase3_gates --gate b --run-dir "$nominal_run" --checkpoint best_nominal --episodes 100 --seed-base 350000
assert _exit_code == 0, 'Gate B failed: do not start CC-FDR.'
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 23. Record a 2v2 match video

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.record_phase3_video --run-dir "$nominal_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_pass_required --episodes 3 --seconds 20 --seed-base 340000
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 24. Record a 3v2 match video

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.record_phase3_video --run-dir "$nominal_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_3v2_press --episodes 3 --seconds 20 --seed-base 341000
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 25. Train CC-FDR from the best nominal checkpoint

In [ ]:
nominal_run = phase3_stage_run('stage_d')
nominal_checkpoint = nominal_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_cc_fdr.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --warm-start "$nominal_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 26. Evaluate CC-FDR

In [ ]:
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.evaluate_phase3 --run-dir "$cc_fdr_run" --checkpoint best_composite --simulator pymunk --scenario phase3_2v2_pass_required --episodes 100 --seed-base 360000 --prefix cc_fdr_pass_required
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 27. Compare recurrent nominal with CC-FDR

In [ ]:
nominal_run = phase3_stage_run('stage_d')
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.evaluate_phase3_gates --gate c --nominal-run "$nominal_run" --cc-fdr-run "$cc_fdr_run" --episodes 100 --seed-base 370000
gate_c_path = cc_fdr_run / 'eval' / 'phase3_gate_c.json'

# 28. Run Gate C

In [ ]:
gate_c = json.loads(gate_c_path.read_text())
display(gate_c['checks'])
assert gate_c['passed'], 'Gate C failed: final confirmation remains disabled.'

# 29. Record robustness and failure videos

In [ ]:
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.record_phase3_video --run-dir "$cc_fdr_run" --checkpoint best_composite --simulator pymunk --scenario phase3_3v2_press --profile combined --episodes 3 --seconds 20 --seed-base 380000
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 30. Update and compile reports

In [ ]:
!python -u -m scripts.export_phase2_report
if shutil.which('latexmk'):
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex

# 31. Sync all artifacts to Drive

In [ ]:
from robosoccer.artifacts import sync_workspace_to_drive
sync_workspace_to_drive(DRIVE_PROJECT, REPO_DIR)

# 32. Optional final three-seed confirmation section, disabled by default

In [ ]:
if not FINAL_CONFIRMATION_ENABLED:
    print('Disabled. Enable only after saved Gates A, B, and C all pass; seed 3 is never confirmation.')
else:
    gate_a = json.loads(CALIBRATION_SUMMARY.read_text())
    gate_b = json.loads((nominal_run / 'eval' / 'phase3_gate_b.json').read_text())
    gate_c = json.loads((cc_fdr_run / 'eval' / 'phase3_gate_c.json').read_text())
    assert gate_a['training_authorized'] and gate_b['passed'] and gate_c['passed']
    for seed in [4, 5, 6]:
        print(f'SUPERVISED JOB seed {seed}: rerun Stage A--D nominal, then CC-FDR and locked evaluations with --seed {seed}.')
    print('Launch each seed as a separate supervised job; this notebook never starts all three automatically.')

# 33. Optional future-platform feasibility notes

In [ ]:
print('A RoboCup 2D/Webots bridge is future work only after calibrated team play, recurrent nominal competence, CC-FDR non-inferiority, and sustained 3v2 video evidence.')

# 34. Finished note

In [ ]:
print('Phase 3 workflow finished. Report only gates backed by saved JSON artifacts; smoke runs are not scientific results.')